# Convert Word/Excel to PDF

In [ ]:
import os
from pathlib import Path
import subprocess
import openpyxl
from openpyxl.utils.dataframe import dataframe_to_rows
import pandas as pd
from reportlab.lib.pagesizes import letter, A4
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors

# Check for LibreOffice availability (Linux alternative)
def check_libreoffice():
    try:
        result = subprocess.run(['libreoffice', '--version'], 
                              capture_output=True, text=True, timeout=5)
        return result.returncode == 0
    except (subprocess.TimeoutExpired, FileNotFoundError):
        return False

LIBREOFFICE_AVAILABLE = check_libreoffice()
if not LIBREOFFICE_AVAILABLE:
    print("LibreOffice not found. Install with: sudo apt-get install libreoffice")

# For Excel to PDF conversion
try:
    EXCEL_PDF_AVAILABLE = True
except ImportError:
    EXCEL_PDF_AVAILABLE = False
    print("Required packages not available. Install with: pip install openpyxl pandas reportlab")

def convert_word_to_pdf(word_file_path, output_pdf_path=None):
    """
    Convert Word document to PDF using LibreOffice (Linux compatible)
    """
    if not LIBREOFFICE_AVAILABLE:
        print("LibreOffice is required for Word to PDF conversion on Linux")
        return False
    
    try:
        word_file_path = Path(word_file_path)
        if output_pdf_path is None:
            output_pdf_path = word_file_path.with_suffix('.pdf')
        else:
            output_pdf_path = Path(output_pdf_path)
        
        # Use LibreOffice headless mode for conversion
        cmd = [
            'libreoffice',
            '--headless',
            '--convert-to', 'pdf',
            '--outdir', str(output_pdf_path.parent),
            str(word_file_path)
        ]
        
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        
        if result.returncode == 0:
            # LibreOffice creates PDF with same name as input file
            generated_pdf = output_pdf_path.parent / f"{word_file_path.stem}.pdf"
            if generated_pdf != output_pdf_path and generated_pdf.exists():
                generated_pdf.rename(output_pdf_path)
            
            print(f"Successfully converted {word_file_path} to {output_pdf_path}")
            return True
        else:
            print(f"LibreOffice conversion failed: {result.stderr}")
            return False
            
    except subprocess.TimeoutExpired:
        print("LibreOffice conversion timed out")
        return False
    except Exception as e:
        print(f"Error converting Word to PDF: {e}")
        return False

def convert_excel_to_pdf(excel_file_path, output_pdf_path=None):
    """
    Convert Excel file to PDF maintaining table structure
    """
    if not EXCEL_PDF_AVAILABLE:
        print("Required packages not available for Excel to PDF conversion")
        return False
    
    try:
        if output_pdf_path is None:
            output_pdf_path = str(Path(excel_file_path).with_suffix('.pdf'))
        
        # Read Excel file
        df = pd.read_excel(excel_file_path, sheet_name=None)  # Read all sheets
        
        # Create PDF document
        doc = SimpleDocTemplate(output_pdf_path, pagesize=A4)
        elements = []
        styles = getSampleStyleSheet()
        
        for sheet_name, sheet_df in df.items():
            # Add sheet title
            title = Paragraph(f"<b>{sheet_name}</b>", styles['Heading1'])
            elements.append(title)
            elements.append(Spacer(1, 12))
            
            # Convert dataframe to table data
            table_data = [list(sheet_df.columns)]  # Headers
            for row in sheet_df.values:
                table_data.append([str(cell) if pd.notna(cell) else '' for cell in row])
            
            # Create table
            table = Table(table_data)
            table.setStyle(TableStyle([
                ('BACKGROUND', (0, 0), (-1, 0), colors.grey),
                ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
                ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
                ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
                ('FONTSIZE', (0, 0), (-1, 0), 10),
                ('BOTTOMPADDING', (0, 0), (-1, 0), 12),
                ('BACKGROUND', (0, 1), (-1, -1), colors.beige),
                ('GRID', (0, 0), (-1, -1), 1, colors.black)
            ]))
            
            elements.append(table)
            elements.append(Spacer(1, 20))
        
        # Build PDF
        doc.build(elements)
        print(f"Successfully converted {excel_file_path} to {output_pdf_path}")
        return True
    except Exception as e:
        print(f"Error converting Excel to PDF: {e}")
        return False

def convert_file_to_pdf(file_path, output_pdf_path=None):
    """
    Auto-detect file type and convert to PDF
    """
    file_path = Path(file_path)
    file_extension = file_path.suffix.lower()
    
    if file_extension in ['.docx', '.doc']:
        return convert_word_to_pdf(file_path, output_pdf_path)
    elif file_extension in ['.xlsx', '.xls']:
        return convert_excel_to_pdf(file_path, output_pdf_path)
    else:
        print(f"Unsupported file format: {file_extension}")
        return False

# Example usage:
# convert_word_to_pdf('document.docx', 'output.pdf')
# convert_excel_to_pdf('spreadsheet.xlsx', 'output.pdf')
# convert_file_to_pdf('file.docx')  # Auto-detect and convert

# Test parsing/extracting with LLM

In [2]:
import os
import json
import time
import subprocess
import tempfile
from pathlib import Path
from typing import List, Optional
import google.generativeai as genai
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
import warnings
from config import GOOGLE_API_KEY

# Imports for conversion (tích hợp từ file_converter.py)
import pandas as pd
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors

warnings.filterwarnings("ignore", category=UserWarning, module='unstructured')

class DocumentParser:
    """
    Document Parser với conversion support nâng cao cho Gemini.
    Tích hợp file_converter để hỗ trợ nhiều format conversion.
    """
    
    def __init__(self, chunk_size: int = 1000, chunk_overlap: int = 150):
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len,
        )
        
        # Rate limiting settings
        self.last_api_call = 0
        self.min_delay_between_calls = 10.0  # Tăng từ 5 lên 10 giây
        self.max_retries = 5  # Tăng số lần thử lại
        self.base_retry_delay = 15.0  # Tăng thời gian chờ cơ bản
        self.exponential_backoff = True  # Thêm backoff theo cấp số nhân
        
        # Thêm giới hạn số lượng request mỗi phút
        self.max_requests_per_minute = 30
        self.request_count = 0
        self.minute_window_start = time.time()
        
        # API Setup
        if not GOOGLE_API_KEY:
            raise ValueError("GOOGLE_API_KEY không được tìm thấy trong config.py")
        
        genai.configure(api_key=GOOGLE_API_KEY)
        
        try:
            self.llm_client = genai.GenerativeModel('gemini-2.0-flash')
            print("✅ Gemini client (gemini-2.0-flash) for parsing đã sẵn sàng.")
        except Exception as e:
            print(f"⚠️ Fallback to gemini-1.5-flash: {e}")
            self.llm_client = genai.GenerativeModel('gemini-1.5-flash')

        # Supported file types cho direct processing
        self.supported_direct_types = {".pdf", ".txt", ".png", ".jpg", ".jpeg", ".gif", ".webp"}
        self.convertible_types = {".docx", ".doc", ".xlsx", ".xls"}
        
        # Check system capabilities
        self.libreoffice_available = self._check_libreoffice()
        self.excel_pdf_available = self._check_excel_pdf_libs()
        
        print(f"🔧 System capabilities:")
        print(f"   - LibreOffice: {'✅' if self.libreoffice_available else '❌'}")
        print(f"   - Excel-to-PDF: {'✅' if self.excel_pdf_available else '❌'}")

    def _check_libreoffice(self) -> bool:
        """Check if LibreOffice is available for document conversion."""
        try:
            result = subprocess.run(['libreoffice', '--version'], 
                                  capture_output=True, text=True, timeout=5)
            return result.returncode == 0
        except (subprocess.TimeoutExpired, FileNotFoundError):
            return False

    def _check_excel_pdf_libs(self) -> bool:
        """Check if required libraries for Excel-to-PDF conversion are available."""
        try:
            import pandas as pd
            from reportlab.lib.pagesizes import A4
            return True
        except ImportError:
            return False

    def _convert_to_supported_format(self, file_path: Path) -> Path:
        """
        Convert unsupported files to supported format using enhanced converters.
        """
        file_extension = file_path.suffix.lower()
        
        if file_extension in [".docx", ".doc"]:
            return self._convert_word_to_pdf_enhanced(file_path)
        elif file_extension in [".xlsx", ".xls"]:
            return self._convert_excel_to_pdf_enhanced(file_path)
        else:
            raise ValueError(f"Conversion not supported for {file_extension}")

    def _convert_word_to_pdf_enhanced(self, word_path: Path) -> Path:
        """
        Enhanced Word to PDF conversion with LibreOffice support.
        """
        print(f"🔄 Converting Word to PDF: {word_path.name}")
        
        # Create temporary PDF file
        with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as tmp_file:
            pdf_path = Path(tmp_file.name)
        
        # Method 1: Try LibreOffice (preferred for Linux)
        if self.libreoffice_available:
            try:
                cmd = [
                    'libreoffice',
                    '--headless',
                    '--convert-to', 'pdf',
                    '--outdir', str(pdf_path.parent),
                    str(word_path)
                ]
                
                result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
                
                if result.returncode == 0:
                    # LibreOffice creates PDF with same name as input file
                    generated_pdf = pdf_path.parent / f"{word_path.stem}.pdf"
                    if generated_pdf.exists():
                        if generated_pdf != pdf_path:
                            generated_pdf.rename(pdf_path)
                        print(f"✅ LibreOffice conversion successful: {pdf_path}")
                        return pdf_path
                    
            except subprocess.TimeoutExpired:
                print("⚠️ LibreOffice conversion timed out")
            except Exception as e:
                print(f"⚠️ LibreOffice conversion failed: {e}")
        
        # Method 2: Fallback to text extraction
        print("🔄 Falling back to text extraction...")
        return self._extract_docx_text_to_temp_file(word_path)

    def _convert_excel_to_pdf_enhanced(self, excel_path: Path) -> Path:
        """
        Enhanced Excel to PDF conversion maintaining table structure.
        """
        print(f"🔄 Converting Excel to PDF: {excel_path.name}")
        
        if self.excel_pdf_available:
            try:
                # Create temporary PDF file
                with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as tmp_file:
                    pdf_path = Path(tmp_file.name)
                
                # Read Excel file
                df_dict = pd.read_excel(excel_path, sheet_name=None)  # Read all sheets
                
                # Create PDF document
                doc = SimpleDocTemplate(str(pdf_path), pagesize=A4)
                elements = []
                styles = getSampleStyleSheet()
                
                for sheet_name, sheet_df in df_dict.items():
                    # Add sheet title
                    title = Paragraph(f"<b>{sheet_name}</b>", styles['Heading1'])
                    elements.append(title)
                    elements.append(Spacer(1, 12))
                    
                    # Convert dataframe to table data
                    if not sheet_df.empty:
                        table_data = [list(sheet_df.columns)]  # Headers
                        for row in sheet_df.values:
                            table_data.append([str(cell) if pd.notna(cell) else '' for cell in row])
                        
                        # Create table with styling
                        table = Table(table_data)
                        table.setStyle(TableStyle([
                            ('BACKGROUND', (0, 0), (-1, 0), colors.grey),
                            ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
                            ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
                            ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
                            ('FONTSIZE', (0, 0), (-1, 0), 8),
                            ('BOTTOMPADDING', (0, 0), (-1, 0), 12),
                            ('BACKGROUND', (0, 1), (-1, -1), colors.beige),
                            ('GRID', (0, 0), (-1, -1), 1, colors.black)
                        ]))
                        
                        elements.append(table)
                        elements.append(Spacer(1, 20))
                
                # Build PDF
                doc.build(elements)
                print(f"✅ Excel-to-PDF conversion successful: {pdf_path}")
                return pdf_path
                
            except Exception as e:
                print(f"⚠️ Excel-to-PDF conversion failed: {e}")
        
        # Fallback to text extraction
        print("🔄 Falling back to text extraction...")
        return self._extract_xlsx_text_to_temp_file(excel_path)

    def _extract_docx_text_to_temp_file(self, docx_path: Path) -> Path:
        """
        Extract text from DOCX and save to temporary text file.
        """
        print(f"🔄 Extracting text from DOCX: {docx_path.name}")
        
        try:
            from docx import Document as DocxDocument
            
            doc = DocxDocument(docx_path)
            full_text = []
            
            # Extract paragraphs
            for para in doc.paragraphs:
                if para.text.strip():
                    full_text.append(para.text)
            
            # Extract tables
            for table in doc.tables:
                table_text = []
                for row in table.rows:
                    row_text = " | ".join(cell.text.strip() for cell in row.cells)
                    if row_text.strip():
                        table_text.append(row_text)
                if table_text:
                    full_text.extend(table_text)
                    full_text.append("")  # Empty line after table
            
            # Save to temporary text file
            with tempfile.NamedTemporaryFile(mode='w', suffix=".txt", delete=False, encoding='utf-8') as tmp_file:
                tmp_file.write('\n'.join(full_text))
                temp_path = Path(tmp_file.name)
            
            print(f"✅ Extracted to text file: {temp_path}")
            return temp_path
            
        except Exception as e:
            print(f"❌ Text extraction failed: {e}")
            raise

    def _extract_xlsx_text_to_temp_file(self, xlsx_path: Path) -> Path:
        """
        Extract data from XLSX and save to temporary text file.
        """
        print(f"🔄 Extracting data from XLSX: {xlsx_path.name}")
        
        try:
            # Read all sheets
            excel_file = pd.ExcelFile(xlsx_path)
            full_text = []
            
            for sheet_name in excel_file.sheet_names:
                df = pd.read_excel(excel_file, sheet_name=sheet_name)
                if not df.empty:
                    full_text.append(f"=== SHEET: {sheet_name} ===")
                    
                    # Convert to markdown-style table
                    headers = " | ".join(str(col) for col in df.columns)
                    separator = " | ".join("---" for _ in df.columns)
                    full_text.append(headers)
                    full_text.append(separator)
                    
                    for _, row in df.iterrows():
                        row_text = " | ".join(str(cell) if pd.notna(cell) else "" for cell in row)
                        full_text.append(row_text)
                    
                    full_text.append("")  # Empty line between sheets
            
            # Save to temporary text file
            with tempfile.NamedTemporaryFile(mode='w', suffix=".txt", delete=False, encoding='utf-8') as tmp_file:
                tmp_file.write('\n'.join(full_text))
                temp_path = Path(tmp_file.name)
            
            print(f"✅ Extracted to text file: {temp_path}")
            return temp_path
            
        except Exception as e:
            print(f"❌ XLSX extraction failed: {e}")
            raise

    # def _apply_rate_limit(self):
    #     """Apply rate limiting to avoid quota exceeded errors."""
    #     current_time = time.time()
    #     time_since_last_call = current_time - self.last_api_call
        
    #     if time_since_last_call < self.min_delay_between_calls:
    #         delay = self.min_delay_between_calls - time_since_last_call
    #         print(f"⏳ Rate limiting: Đợi {delay:.1f} giây...")
    #         time.sleep(delay)
        
    #     self.last_api_call = time.time()
    
    
    def _apply_rate_limit(self):
        """Enhanced rate limiting with multiple strategies."""
        current_time = time.time()
        
        # 1. Basic delay between calls
        time_since_last_call = current_time - self.last_api_call
        if time_since_last_call < self.min_delay_between_calls:
            delay = self.min_delay_between_calls - time_since_last_call
            print(f"⏳ Basic rate limiting: Waiting {delay:.1f} seconds...")
            time.sleep(delay)
        
        # 2. Requests per minute limit
        if current_time - self.minute_window_start > 60:
            # Reset counter if we're in a new minute window
            self.request_count = 0
            self.minute_window_start = current_time
        else:
            self.request_count += 1
        
        if self.request_count >= self.max_requests_per_minute:
            time_remaining = 60 - (current_time - self.minute_window_start)
            print(f"⏳ Minute limit reached. Waiting {time_remaining:.1f} seconds...")
            time.sleep(time_remaining)
            self.request_count = 0
            self.minute_window_start = time.time()
        
        self.last_api_call = time.time()

    def _parse_with_llm_with_retry(self, file_path: Path) -> str:
        """
        Parse file với enhanced conversion support và retry logic.
        """
        original_path = file_path
        temp_file_created = False
        
        # Check if conversion is needed
        if file_path.suffix.lower() not in self.supported_direct_types:
            if file_path.suffix.lower() in self.convertible_types:
                print(f"📋 File type {file_path.suffix} không được Gemini hỗ trợ trực tiếp. Đang convert...")
                try:
                    file_path = self._convert_to_supported_format(file_path)
                    temp_file_created = True
                except Exception as conv_error:
                    print(f"❌ Conversion failed: {conv_error}")
                    return ""
            else:
                print(f"❌ File type {file_path.suffix} không được hỗ trợ.")
                return ""

        print(f"🧠 Bắt đầu parsing bằng LLM cho file: {original_path.name}...")
        
        uploaded_file = None
        
        try:
            for attempt in range(self.max_retries):
                try:
                    # Rate limiting
                    self._apply_rate_limit()
                    
                    # Upload file
                    if not uploaded_file:
                        uploaded_file = genai.upload_file(path=str(file_path), display_name=original_path.name)
                    
                    # Wait for processing
                    max_wait_time = 60
                    wait_time = 0
                    while uploaded_file.state.name == "PROCESSING" and wait_time < max_wait_time:
                        time.sleep(2)
                        wait_time += 2
                        uploaded_file = genai.get_file(uploaded_file.name)
                    
                    if uploaded_file.state.name == "FAILED":
                        print(f"❌ File upload failed: {uploaded_file.state}")
                        return ""
                    
                    # Generate content
                    self._apply_rate_limit()
                    
                    prompt = """
                    <TASK_DEFINITION>
                    You are an automated data processing engine. Your sole task is to analyze the provided file and convert its entire content into a single, clean, and well-structured Markdown string. The output must be a perfect representation of the original data, suitable for machine parsing later.

                    Follow these critical instructions precisely:

                    1.  **Analyze Layout:** First, analyze the visual layout of the document. Identify key-value pairs (e.g., a label in one cell and its value in another, potentially non-adjacent cell) and structured tables.
                    2.  **Convert Key-Value Pairs:** Represent all identified key-value pairs clearly.
                    3.  **Convert Tables:** Convert all structured tables into standard Markdown table format.
                    4.  **Preserve Content:** All text and numerical data must be preserved exactly as it appears in the original file.
                    5.  **No Extra Content:** Do not add any summaries, explanations, comments, or any text that is not present in the original document.
                    6.  **Strict Output Format:** Your entire output must be ONLY the Markdown content. Do not wrap it in ```markdown ... ``` or any other formatting.
                    </TASK_DEFINITION>

                    <OUTPUT_EXAMPLE>
                    ### A. THÔNG TIN CHUNG
                    - **Ngày thực hiện:** 4/16/2024
                    - **CusID:** 22079986
                    - **Tên Khách hàng:** CÔNG TY CỔ PHẦN MẶT DỰNG CAG
                    - **Phân khúc:** MM
                    - **Subsegment:** Dịch vụ Xây lắp, lắp đặt
                    - **XHTD:** Aa3
                    - **BBC:** (Trống)
                    - **DDA:** Vùng
                    </OUTPUT_EXAMPLE>

                    Analyze the provided file. Think step-by-step to ensure all data is captured accurately, then generate the final Markdown output.
                    """


                    response = self.llm_client.generate_content([uploaded_file, prompt])
                    print(f"✅ LLM đã parse thành công file: {original_path.name}")
                    return response.text

                except Exception as e:
                    error_message = str(e)
                    
                    if "429" in error_message or "quota" in error_message.lower():
                        retry_delay = self.base_retry_delay * (2 ** attempt)
                        print(f"  ⚠️ Gặp lỗi Rate Limit. Đang thử lại sau {retry_delay} giây... (Lần {attempt + 1}/{self.max_retries})")
                        time.sleep(retry_delay)
                        continue
                    elif "mimeType" in error_message or "not supported" in error_message:
                        print(f"❌ Lỗi MIME type không được hỗ trợ: {e}")
                        break
                    elif "503" in error_message or "Service Unavailable" in error_message:
                        retry_delay = self.base_retry_delay * (2 ** attempt)
                        print(f"  ⚠️ Gemini service unavailable. Đang thử lại sau {retry_delay} giây... (Lần {attempt + 1}/{self.max_retries})")
                        time.sleep(retry_delay)
                        continue
                    else:
                        print(f"❌ Lỗi khác khi parsing: {e}")
                        break
            
            print(f"  ❌ Đã thử lại {self.max_retries} lần nhưng vẫn gặp lỗi. Bỏ cuộc.")
            return ""
            
        finally:
            # Cleanup
            if uploaded_file:
                try:
                    genai.delete_file(uploaded_file.name)
                    print(f"🧹 Đã cleanup uploaded file: {original_path.name}")
                except:
                    pass
            
            # Remove temporary file if created
            if temp_file_created and file_path.exists():
                try:
                    os.unlink(file_path)
                    print(f"🧹 Đã xóa file tạm: {file_path}")
                except:
                    pass

    def parse_file(self, file_path: str) -> List[Document]:
        """
        Parse file với enhanced conversion support.
        """
        file_path = Path(file_path)
        
        if not file_path.exists():
            print(f"❌ File không tồn tại: {file_path}")
            return []
        
        # Parse với LLM
        full_markdown_content = self._parse_with_llm_with_retry(file_path)
        
        if not full_markdown_content or not full_markdown_content.strip():
            print(f"⚠️ LLM không trả về nội dung nào cho file {file_path.name}. Trả về danh sách rỗng.")
            return []
            
        # Chunking
        # print(f"🔪 Bắt đầu chunking nội dung Markdown (độ dài: {len(full_markdown_content)} chars)...")
        
        # chunks = self.text_splitter.create_documents([full_markdown_content])
        
        # # Add metadata
        # for chunk in chunks:
        #     chunk.metadata = {
        #         "source": str(file_path),
        #         "basename": file_path.name,
        #         "file_type": file_path.suffix.lower(),
        #         "content_type": "llm_parsed_markdown_chunk",
        #         "parser_method": "gemini_enhanced"
        #     }
            
        # print(f"✅ Hoàn tất! Tạo ra {len(chunks)} documents từ file {file_path.name}.")
        # return chunks
        return full_markdown_content

    def convert_file_to_pdf(self, file_path: str, output_pdf_path: Optional[str] = None) -> Optional[str]:
        """
        Public method to convert files to PDF format.
        
        Args:
            file_path: Path to input file
            output_pdf_path: Optional output path for PDF
            
        Returns:
            Path to converted PDF file or None if conversion failed
        """
        file_path = Path(file_path)
        file_extension = file_path.suffix.lower()
        
        if file_extension not in self.convertible_types:
            print(f"❌ Conversion không được hỗ trợ cho file type: {file_extension}")
            return None
        
        try:
            if output_pdf_path:
                output_path = Path(output_pdf_path)
            else:
                output_path = file_path.with_suffix('.pdf')
            
            if file_extension in ['.docx', '.doc']:
                converted_path = self._convert_word_to_pdf_enhanced(file_path)
                print(f"Nội dung của file PDF là : {converted_path}")
            elif file_extension in ['.xlsx', '.xls']:
                converted_path = self._convert_excel_to_pdf_enhanced(file_path)
                print(f"Nội dung của file PDF là : {converted_path}")
            else:
                return None
            
            # Move converted file to desired location if different
            if converted_path != output_path:
                converted_path.rename(output_path)
            
            return str(output_path)
            
        except Exception as e:
            print(f"❌ Conversion failed: {e}")
            return None
        
if __name__ == "__main__":
    parser = DocumentParser()
    parse_folder = Path('/home/locmt/Techcombank_/chatbot_document/data/data_real_new')
    combined_data = []  # Khởi tạo trước vòng lặp
    for files in list(parse_folder.glob('*')):
        if files.suffix.lower() in parser.convertible_types:
            print(f"📄 Đang parse file: {files.name}")
            doc_test = parser.parse_file(files)
            if doc_test:
                print(f"✅ Đã parse thành công: {files.name}")
                combined_data.append(doc_test)  # Thêm chuỗi vào danh sách
            else:
                print(f"❌ Không thể parse file: {files.name}")
    print(f"📄 Đã parse {len(combined_data)} documents từ thư mục {parse_folder.name}.")
    os.makedirs('output_parsing', exist_ok=True)
    with open('output_parsing/parsed_output.md', 'w', encoding='utf-8') as f:
        for doc in combined_data:
            f.write(doc + '\n\n')
    print("✅ Đã lưu kết quả parsing vào file: output_parsing/parsed_output.md")

✅ Gemini client (gemini-2.0-flash) for parsing đã sẵn sàng.
🔧 System capabilities:
   - LibreOffice: ✅
   - Excel-to-PDF: ✅
📄 Đang parse file: MB01-HD_1.xlsx
📋 File type .xlsx không được Gemini hỗ trợ trực tiếp. Đang convert...
🔄 Converting Excel to PDF: MB01-HD_1.xlsx
✅ Excel-to-PDF conversion successful: /tmp/tmple3cq6e6.pdf
🧠 Bắt đầu parsing bằng LLM cho file: MB01-HD_1.xlsx...
⏳ Basic rate limiting: Waiting 6.8 seconds...
✅ LLM đã parse thành công file: MB01-HD_1.xlsx
🧹 Đã cleanup uploaded file: MB01-HD_1.xlsx
🧹 Đã xóa file tạm: /tmp/tmple3cq6e6.pdf
✅ Đã parse thành công: MB01-HD_1.xlsx
📄 Đang parse file: SoSanhDNCungNganh_2.docx
📋 File type .docx không được Gemini hỗ trợ trực tiếp. Đang convert...
🔄 Converting Word to PDF: SoSanhDNCungNganh_2.docx
✅ LibreOffice conversion successful: /tmp/tmppuvlwjec.pdf
🧠 Bắt đầu parsing bằng LLM cho file: SoSanhDNCungNganh_2.docx...
⏳ Basic rate limiting: Waiting 4.9 seconds...
⏳ Basic rate limiting: Waiting 7.2 seconds...
✅ LLM đã parse thành c

In [3]:
combined_data

['### A. THÔNG TIN CHUNG\n- **Ngày thực hiện:** 2024-04-16 00:00:00\n- **CusID:** 22079986\n- **Tên Khách hàng:** CÔNG TY CỔ PHẦN MỘT DỰNG CAG\n- **Phân khúc:** MM\n- **Subsegment:** Dịch vụ Xây lắp, lắp đặt\n- **XHTD:** Aa3\n- **BBC:** BBC\n- **DDA:** DDA\n',
 'Năm\nThành\nlập\n2004\n\nQuy\nmô\ndoanh\nthu\n2023\n944 tỷ đồng\n\nSản\nphẩm\n*   Hệ thống mặt\n    dựng kính\n*   Tấm ốp Alu\n*   Hệ thống cửa\n    kính (cửa tự\n    động, cửa mở\n    quay, cửa đi\n    lùa), cửa số (mở\n    quay, mở hắt)\n*   Lam chắn nắng,\n    cầu thang kính\n',
 'Sheet1\nNi dung site visit\nNhôm thanh:\nCác mã kích th█c nh█: có th■ nhép hàng săn xuất trong n__c\nCác mã kích th_c In: trong nc ko sn xu t => hàng nhép kh█u Trung Quc\nPhôi kính:\nun gc nhép khu, công ty thng mua thông qua cácn và chuyên nhép phôi kính do nu nh p khu trc tips không lá\nKính không màu: có th■ nh p hàng săn xu_t trong n_c\nKính màu: trong nc ko sn xut_c => hàng nh p khu.\nThông thư Ing kính 2 lps\ngm 1 lp màu (bên ngoài & có lép c

# Test Query (not chunking)

In [5]:
with open('output_parsing/parsed_output.md', 'r', encoding='utf-8') as f:
    doc_test = f.read()
    
print(type(doc_test))

def test_query_on_document(query: str, document_content: str) -> str:
    """
    Test a query against the document content using Gemini LLM
    """
    try:
        # Apply rate limiting
        parser._apply_rate_limit()
        
        prompt = f"""
Dựa trên nội dung tài liệu sau:

{document_content}

Hãy trả lời câu hỏi: {query}

Yêu cầu:
- Trả lời chính xác dựa trên thông tin có trong tài liệu
- Nếu không tìm thấy thông tin, hãy nói rõ "Không tìm thấy thông tin này trong tài liệu"
"""

        response = parser.llm_client.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"Lỗi khi xử lý query: {e}"

# Test với một số câu hỏi mẫu
template4_queries = [
    "Tên đầy đủ của khách hàng là gì?",
    "Số giấy phép đăng ký kinh doanh của khách hàng là gì?",
    "ID khách hàng trên hệ thống T24 là gì?",
    "Phân khúc của khách hàng là gì?",
    "Loại khách hàng là gì?",
    "Ngành nghề hoạt động kinh doanh của khách hàng là gì?",
    "Mục đích báo cáo là gì?",
    "Kết quả phân luồng của khách hàng là gì?",
    "XHTD của khách hàng là gì?",
    "Ngày thành lập công ty là ngày nào?",
    "Địa chỉ đăng ký kinh doanh của công ty là gì?",
    "Người đại diện theo pháp luật của công ty là ai?",
    "Khách hàng có kinh doanh ngành nghề có điều kiện không?",
    "Thông tin chi tiết về ban lãnh đạo công ty là gì?",
    "Thông tin chi tiết về đầu vào sản xuất kinh doanh là gì?",
    "Thông tin chi tiết về đầu ra sản xuất kinh doanh là gì?",
    "Nhận xét về thông tin khách hàng là gì?",
    "Nhận xét về pháp lý/GPKD có điều kiện là gì?",
    "Nhận xét về chủ doanh nghiệp/ban lãnh đạo là gì?",
    "Nhận xét về KYC là gì?",
    "Lĩnh vực kinh doanh của công ty là gì?",
    "Sản phẩm/dịch vụ của công ty là gì?",
    "Tỷ trọng doanh thu năm N-1 (%) là bao nhiêu?",
    "Tỷ trọng doanh thu năm N (%) là bao nhiêu?",
    "Nhóm mặt hàng của công ty là gì?",
    "Tỷ trọng doanh thu năm 2023 là bao nhiêu?",
    "Tỷ trọng doanh thu 10T/2024 là bao nhiêu?",
    "Mô tả chung về sản phẩm của công ty là gì?",
    "Mô tả lợi thế cạnh tranh của công ty là gì?",
    "Mô tả năng lực đấu thầu của công ty là gì?",
    "Quy trình vận hành (tóm tắt) của công ty là gì?",
    "Đầu vào - mặt hàng là gì?",
    "Đầu vào - chi tiết là gì?",
    "Đầu vào - phương thức thanh toán là gì?",
    "Đầu ra - kênh phân phối là gì?",
    "Đầu ra - tỷ trọng là bao nhiêu?",
    "Đầu ra - phương thức thanh toán là gì?",
    "Nhận xét tổng quan về hoạt động kinh doanh là gì?",
    "Phân tích cung cầu ngành là gì?",
    "Nhận xét về thông tin ngành là gì?"
]

print("🤖 Bắt đầu test queries trên document...")
print("=" * 60)

for i, query in enumerate(template4_queries, 1):
    print(f"\n📝 Query {i}: {query}")
    print("-" * 40)

    answer = test_query_on_document(query, doc_test)
    print(f"💬 Trả lời: {answer}")
    print("-" * 40)

<class 'str'>
🤖 Bắt đầu test queries trên document...

📝 Query 1: Tên đầy đủ của khách hàng là gì?
----------------------------------------
💬 Trả lời: Tên đầy đủ của khách hàng là CÔNG TY CỔ PHẦN MẶT DỰNG CAG.

----------------------------------------

📝 Query 2: Số giấy phép đăng ký kinh doanh của khách hàng là gì?
----------------------------------------
⏳ Basic rate limiting: Waiting 9.1 seconds...
💬 Trả lời: Số giấy phép đăng ký kinh doanh của khách hàng là: 0101442420

----------------------------------------

📝 Query 3: ID khách hàng trên hệ thống T24 là gì?
----------------------------------------
⏳ Basic rate limiting: Waiting 9.0 seconds...
💬 Trả lời: Không tìm thấy thông tin này trong tài liệu

----------------------------------------

📝 Query 4: Phân khúc của khách hàng là gì?
----------------------------------------
⏳ Basic rate limiting: Waiting 9.2 seconds...
💬 Trả lời: Phân khúc của khách hàng là MM.

----------------------------------------

📝 Query 5: Loại khách hàng l

KeyboardInterrupt: 

40


# Test chunking

In [11]:
import os
from pathlib import Path

# Tạo thư mục output
output_dir = Path("chunks_output")
output_dir.mkdir(exist_ok=True)

print(f"💾 Bắt đầu lưu {len(doc_test)} chunks...")

for i, chunk in enumerate(doc_test, 1):
    # Tạo filename đơn giản với index
    filename = f"chunk_{i:03d}.txt"
    filepath = output_dir / filename
    
    # Lưu content
    try:
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(f"CHUNK {i}\n")
            f.write("=" * 40 + "\n")
            f.write(f"Source: {chunk.metadata.get('basename', 'Unknown')}\n")
            f.write(f"Length: {len(chunk.page_content)} chars\n")
            f.write("=" * 40 + "\n\n")
            f.write(chunk.page_content)
        
        print(f"✅ Chunk {i:3d} → {filename}")
        
    except Exception as e:
        print(f"❌ Lỗi chunk {i}: {e}")

print(f"\n🎉 Hoàn tất! Đã lưu {len(doc_test)} chunks vào: {output_dir}")

💾 Bắt đầu lưu 5 chunks...
✅ Chunk   1 → chunk_001.txt
✅ Chunk   2 → chunk_002.txt
✅ Chunk   3 → chunk_003.txt
✅ Chunk   4 → chunk_004.txt
✅ Chunk   5 → chunk_005.txt

🎉 Hoàn tất! Đã lưu 5 chunks vào: chunks_output
